In [1]:
import os
import subprocess
from tqdm.notebook import tqdm

VIDEO_DIR = "/home/cyl476530/ReKV/data/StreamingBench/src/data/videos"
REPAIR_DIR = os.path.join(VIDEO_DIR, "repaired")
os.makedirs(REPAIR_DIR, exist_ok=True)

def check_video_ffmpeg(video_path):
    """
    利用ffmpeg检测视频是否损坏。
    """
    try:
        cmd = [
            "ffmpeg", "-v", "error", "-i", video_path, "-f", "null", "-"
        ]
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=30)
        errors = result.stderr.decode()
        if ("Invalid NAL unit size" in errors or
            "Error splitting the input into NAL units" in errors or
            "error" in errors.lower() or
            "failed" in errors.lower()):
            return False, errors
        return True, ""
    except Exception as e:
        return False, str(e)

def repair_video(video_path, output_path):
    """
    ffmpeg重封装/转码修复
    """
    try:
        cmd = [
            "ffmpeg", "-err_detect", "ignore_err", "-i", video_path,
            "-c:v", "libx264", "-c:a", "copy", "-y", output_path
        ]
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=600)
        return True, result.stderr.decode()
    except Exception as e:
        return False, str(e)


In [2]:
exts = (".mp4", ".avi", ".mkv", ".mov", ".flv", ".webm")
video_files = []
for root, _, files in os.walk(VIDEO_DIR):
    for f in files:
        if f.lower().endswith(exts):
            video_files.append(os.path.join(root, f))
print(f"共检测到 {len(video_files)} 个视频。")


共检测到 900 个视频。


In [3]:
bad_videos = []

for video in tqdm(video_files, desc="检测视频"):
    ok, err = check_video_ffmpeg(video)
    if not ok:
        print(f"异常: {video}\n错误信息摘要: {err[:200]}")
        bad_videos.append((video, err))
        repaired_path = os.path.join(REPAIR_DIR, os.path.basename(video))
        repair_ok, repair_log = repair_video(video, repaired_path)
        if repair_ok:
            print(f"已修复: {video}\n新文件: {repaired_path}")
        else:
            print(f"修复失败: {video}\n原因: {repair_log[:200]}")


检测视频:   0%|          | 0/900 [00:00<?, ?it/s]

异常: /home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_12_Multimodal_Alignment.mp4
错误信息摘要: Command '['ffmpeg', '-v', 'error', '-i', '/home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_12_Multimodal_Alignment.mp4', '-f', 'null', '-']' timed out after 30 seconds
已修复: /home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_12_Multimodal_Alignment.mp4
新文件: /home/cyl476530/ReKV/data/StreamingBench/src/data/videos/repaired/sample_12_Multimodal_Alignment.mp4
异常: /home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_188_real.mp4
错误信息摘要: Command '['ffmpeg', '-v', 'error', '-i', '/home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_188_real.mp4', '-f', 'null', '-']' timed out after 30 seconds
已修复: /home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_188_real.mp4
新文件: /home/cyl476530/ReKV/data/StreamingBench/src/data/videos/repaired/sample_188_real.mp4
异常: /home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_189_real.mp4

In [4]:
if bad_videos:
    print(f"共有 {len(bad_videos)} 个损坏或异常视频：")
    for path, err in bad_videos:
        print(f"{path}\n------错误摘要：\n{err[:200]}\n")
else:
    print("所有视频解码检测均正常。")

共有 56 个损坏或异常视频：
/home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_12_Multimodal_Alignment.mp4
------错误摘要：
Command '['ffmpeg', '-v', 'error', '-i', '/home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_12_Multimodal_Alignment.mp4', '-f', 'null', '-']' timed out after 30 seconds

/home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_188_real.mp4
------错误摘要：
Command '['ffmpeg', '-v', 'error', '-i', '/home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_188_real.mp4', '-f', 'null', '-']' timed out after 30 seconds

/home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_189_real.mp4
------错误摘要：
Command '['ffmpeg', '-v', 'error', '-i', '/home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_189_real.mp4', '-f', 'null', '-']' timed out after 30 seconds

/home/cyl476530/ReKV/data/StreamingBench/src/data/videos/sample_190_real.mp4
------错误摘要：
Command '['ffmpeg', '-v', 'error', '-i', '/home/cyl476530/ReKV/data/StreamingBench/src/data